In [1]:
!pip install requests

In [ ]:
import requests
import pandas as pd
import json

# --- 1. Load the list of NORAD_CAT_IDs from your dataset ---
df = pd.read_csv('space_debris_with_engineered_features.csv')
norad_ids = df['NORAD_CAT_ID'].unique().tolist()
print(f"Found {len(norad_ids)} unique NORAD IDs in your dataset.")

# --- 2. Space-Track.org API credentials and endpoints ---
username = 'ayushrajujaiswal@gmail.com'  # Replace with your Space-Track.org username
password = 'rraa4@korutla'  # Replace with your Space-Track.org password

login_url = 'https://www.space-track.org/ajaxauth/login'
query_url = 'https://www.space-track.org/basicspacedata/query/class/gp/NORAD_CAT_ID/{ids}/orderby/EPOCH%20desc/format/tle'

# --- 3. Authenticate and get TLEs ---
try:
    # Log in to the API
    session = requests.Session()
    session.post(login_url, data={'identity': username, 'password': password})

    # Check if login was successful
    if not session.cookies.get('PHPSESSID'):
        raise Exception("Login failed. Check your username and password.")

    # Split the list of NORAD IDs into chunks to avoid URL length limits
    chunk_size = 200
    id_chunks = [norad_ids[i:i + chunk_size] for i in range(0, len(norad_ids), chunk_size)]

    tle_data_list = []

    for i, chunk in enumerate(id_chunks):
        ids_string = ','.join(map(str, chunk))
        response = session.get(query_url.format(ids=ids_string))

        if response.status_code == 200:
            tle_data_list.append(response.text)
        else:
            print(f"Error for chunk {i}: {response.status_code}")
            print(response.text)

    # Combine and save all TLE data
    all_tle_data = ''.join(tle_data_list)

    with open('debris_tles.txt', 'w') as f:
        f.write(all_tle_data)

    print(f"\nSuccessfully downloaded TLE data for {len(norad_ids)} IDs.")
    print("Data saved to 'debris_tles.txt'")

except Exception as e:
    print(f"An error occurred: {e}")